In [1]:
import pandas as pd 
import numpy as np
from sklearn.preprocessing import StandardScaler

In [2]:
def read_and_merge(fpkm_path, gistic_path, rppa_path, labels_path):
    fpkm = pd.read_csv(fpkm_path, index_col=0)
    gistic = pd.read_csv(gistic_path, index_col=0)
    rppa = pd.read_csv(rppa_path, index_col=0)
    labels = pd.read_csv(labels_path, index_col=0)

    # Ensure Sample_ID is index; allow file with Sample_ID as first column
    for df in (fpkm, gistic, rppa, labels):
        if 'Sample_ID' in df.columns:
            df.set_index('Sample_ID', inplace=True)

    # intersection of samples
    samples = set(fpkm.index) & set(gistic.index) & set(rppa.index) & set(labels.index)
    samples = sorted(samples)
    if len(samples) == 0:
        raise ValueError('No overlapping Sample_ID between files')

    fpkm = fpkm.loc[samples]
    gistic = gistic.loc[samples]
    rppa = rppa.loc[samples]
    labels = labels.loc[samples]

    return fpkm, gistic, rppa, labels

In [3]:
def preprocess_fpkm(fpkm_df, gene_subset=None):
    # log2(x+1)
    X = np.log2(fpkm_df.values.astype(float) + 1.0)
    # X = fpkm_df.values.astype(float)
    genes = list(fpkm_df.columns)
    sample_ids = list(fpkm_df.index)
    if gene_subset is not None:
        keep = [g for g in gene_subset if g in genes]
        if len(keep) == 0:
            raise ValueError('No genes from subset found in FPKM')
        idx = [genes.index(g) for g in keep]
        X = X[:, idx]
        genes = keep
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    return X, genes, scaler, sample_ids


def preprocess_rppa(rppa_df, protein_subset=None):
    X = rppa_df.values.astype(float)
    proteins = list(rppa_df.columns)
    sample_ids = list(rppa_df.index)
    if protein_subset is not None:
        keep = [p for p in protein_subset if p in proteins]
        idx = [proteins.index(p) for p in keep]
        X = X[:, idx]
        proteins = keep
    scaler = StandardScaler()
    X = scaler.fit_transform(X)
    return X, proteins, scaler, sample_ids


def preprocess_gistic(gistic_df, gene_subset=None, two_channel=True):
    # gistic values expected -1,0,1
    G = gistic_df.values.astype(int)
    genes = list(gistic_df.columns)
    sample_ids = list(gistic_df.index)
    if gene_subset is not None:
        keep = [g for g in gene_subset if g in genes]
        idx = [genes.index(g) for g in keep]
        G = G[:, idx]
        genes = keep
    if two_channel:
        # channel 0 = gain, channel1 = loss
        gain = (G == 1).astype(float)
        loss = (G == -1).astype(float)
        X = np.concatenate([gain, loss], axis=1)
        col_names = [g + '_gain' for g in genes] + [g + '_loss' for g in genes]
    else:
        X = G.astype(float)
        col_names = genes
    scaler = None
    print(X.shape)
    return X, col_names, scaler, sample_ids

In [12]:
fpkm_path = '/home3/tuannd/Cancer-Multi-Omics-Benchmark/data__COADREAD/fpkm_data.csv'
gistic_path = '/home3/tuannd/Cancer-Multi-Omics-Benchmark/data__COADREAD/gistic_data.csv'
rppa_path = '/home3/tuannd/Cancer-Multi-Omics-Benchmark/data__COADREAD/rppa_data.csv'
labels_path = '/home3/tuannd/Cancer-Multi-Omics-Benchmark/data__COADREAD/sample_classes.csv'
fpkm_df, gistic_df, rppa_df, labels_df = read_and_merge(fpkm_path, gistic_path, rppa_path, labels_path)

In [13]:
# print the stats of labels
labels_df['class'].value_counts()

class
0    86
1    69
2    34
3    33
Name: count, dtype: int64

In [14]:
print(f'mRNA features: {fpkm_df.shape[1]}')
print(f'CNV features: {gistic_df.shape[1]}')
print(f'RPPA features: {rppa_df.shape[1]}')

mRNA features: 20530
CNV features: 24776
RPPA features: 153


In [ ]:
X_rna, genes, scaler_rna, sample_ids = preprocess_fpkm(fpkm_df)

In [ ]:
# new dataframe format
# each column is a sample, each row is a gene
# row index is the gene name
# column index is the sample name
# value is the expression leveal
data_dict = {}
for sample_id, values in zip(sample_ids, X_rna):
    data_dict[sample_id] = values

df_rna = pd.DataFrame(data_dict, index=genes)
df_rna.head()

In [ ]:
df_rna.shape

In [ ]:
# df_rna.to_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/csv/processed/COADREAD/COADREAD_mRNA.csv')

In [ ]:
X_rppa, proteins, scaler_rppa, sample_ids = preprocess_rppa(rppa_df)
data_dict = {}
for sample_id, values in zip(sample_ids, X_rppa):
    data_dict[sample_id] = values

df_rppa = pd.DataFrame(data_dict, index=proteins)
df_rppa.head()

In [ ]:
X_rppa.shape

In [ ]:
# df_rppa.to_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/csv/processed/COADREAD/COADREAD_proteomics.csv')

In [ ]:
X_cnv, cnv_cols, _, sample_ids = preprocess_gistic(gistic_df, two_channel=False)
data_dict = {}
for sample_id, values in zip(sample_ids, X_cnv):
    data_dict[sample_id] = values

df_cnv = pd.DataFrame(data_dict, index=cnv_cols)
df_cnv.head()

In [ ]:
df_cnv.to_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/csv/processed/COADREAD/COADREAD_CNV_1_channel.csv')

In [ ]:
label = pd.read_csv(labels_path)
label.head()

In [ ]:
label['class'].value_counts()

In [ ]:
label['Label'] = label['class']
label['Label'].to_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/csv/processed/COADREAD/COADREAD_label_num.csv', index=False)

### Perform GSEA

In [ ]:
import pandas as pd 

In [ ]:
df_label = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Top/BRCA_label_num.csv')
len(df_label)

In [ ]:

df_mirna = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Original/BRCA_miRNA.csv')

In [ ]:
len(df_mirna)

In [ ]:
df_mirna

In [ ]:
df_mrna = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Original/BRCA_mRNA.csv')
df_mrna

In [ ]:
from typing import Dict, List

def load_gmt(gmt_path: str) -> Dict[str, List[str]]:
	"""Load a GMT file into a dict: gene_set_name -> list of genes.

	Each line: NAME <tab> URL/desc <tab> GENE1 <tab> GENE2 ...
	"""
	gene_sets: Dict[str, List[str]] = {}
	with open(gmt_path, "r", encoding="utf-8") as f:
		for line in f:
			parts = line.strip().split("\t")
			if not parts:
				continue
			name = parts[0]
			genes = [g for g in parts[2:] if g]
			gene_sets[name] = genes
	return gene_sets

In [ ]:
gmt_data_path = '/home3/tuannd/Cancer-Multi-Omics-Benchmark/gsea_data/h.all.v2025.1.Hs.symbols.gmt'
gene_sets = load_gmt(gmt_data_path)
gene_sets

In [ ]:
import numpy as np
import math
from typing import Dict, List, Tuple

def preprocess_gene_list(gene_to_value: Dict[str, float]) -> Tuple[List[str], np.ndarray]:
	"""Preprocess gene list following DeepCC's preprocessGeneList R function.
	
	- Drop NA names/values
	- Aggregate duplicate genes by max
	- Sort decreasing by value
	
	Args:
		gene_to_value: Dictionary mapping gene names to expression values
		
	Returns:
		ordered_genes: List of gene names sorted by decreasing value
		values: numpy array of values aligned with ordered_genes
	"""
	# Drop invalids and aggregate by max
	agg: Dict[str, float] = {}
	for g, v in gene_to_value.items():
		if g is None or g == "":
			continue
		if v is None or (isinstance(v, float) and math.isnan(v)):
			continue
		prev = agg.get(g)
		if prev is None or v > prev: # keep the highest value
			agg[g] = float(v)
	
	# Sort by decreasing value
	sorted_items = sorted(agg.items(), key=lambda kv: kv[1], reverse=True)
	genes = [g for g, _ in sorted_items]
	values = np.array([v for _, v in sorted_items], dtype=np.float32)
	return genes, values

In [ ]:
from typing import Iterable

def calc_enrichment_score(ordered_genes: List[str], values: np.ndarray, gene_set: Iterable[str], exponent: float = 1.0) -> float:
	"""Calculate enrichment score following DeepCC's calcEnrichmentScoreCPP C++ implementation.
	
	This implements the GSEA-like enrichment score calculation used in DeepCC.
	
	Args:
		ordered_genes: List of genes after preprocessing (descending by values)
		values: numpy array aligned to ordered_genes with expression values
		gene_set: Iterable of genes in the gene set
		exponent: Exponent for weighting (default: 1.0)
		
	Returns:
		ES: Enrichment score (float)
	"""
	N = len(ordered_genes)
	if N == 0:
		return 0.0
	
	set_lookup = set(gene_set)
	sset = np.array([1 if g in set_lookup else 0 for g in ordered_genes], dtype=np.float32)
	nh = int(np.sum(sset))
	
	if nh == 0:
		return 0.0
	
	# nr = sum(|value|^exp over hits)
	abs_vals = np.abs(values)
	hit_weights = np.power(abs_vals, exponent) * sset
	nr = np.sum(hit_weights)
	
	if nr == 0.0:
		return 0.0
	
	n_miss = N - nh
	n = -1.0 / float(n_miss) if n_miss > 0 else 0.0
	
	cs = 0.0
	smax = 0.0
	smin = 0.0
	
	for j in range(N):
		if sset[j] > 0:
			cs += float((abs_vals[j] ** exponent) / nr)
		else:
			cs += n
		if cs > smax:
			smax = cs
		elif cs < smin:
			smin = cs
	
	return smax if abs(smax) > abs(smin) else smin


In [ ]:
# Check the structure of df_mrna
print(f"mRNA DataFrame shape: {df_mrna.shape}")
print(f"First few rows/columns:")
print(df_mrna.head())
print(f"\nIndex (likely genes): {df_mrna.index[:5].tolist()}")
print(f"Columns (likely samples): {df_mrna.columns[:5].tolist()}")

In [ ]:
from tqdm import tqdm
def get_functional_spectra(expression_df: pd.DataFrame, gene_sets: Dict[str, List[str]], scale: bool = True) -> pd.DataFrame:
	"""Generate functional spectra for a batch of samples following DeepCC's getFunctionalSpectra.
	
	Args:
		expression_df: DataFrame with genes as columns and samples as rows
		gene_sets: Dictionary mapping gene set names to lists of genes
		scale: Whether to center columns (subtract column means)
		
	Returns:
		DataFrame with rows=samples, cols=gene set names (enrichment scores)
	"""
	if scale:
		# Center columns (subtract column means)
		col_means = expression_df.mean(axis=0)
		expression_df = expression_df.subtract(col_means, axis=0)
	
	set_names = list(gene_sets.keys())
	result = np.zeros((expression_df.shape[0], len(set_names)), dtype=np.float32)
	
	for i, (sample_id, row) in tqdm(enumerate(expression_df.iterrows()), total=expression_df.shape[0]):
		# Convert row to gene->value dictionary
		gene_to_val = {g: float(v) for g, v in row.items()}
		ordered_genes, values = preprocess_gene_list(gene_to_val)
		
		# Calculate enrichment score for each gene set
		for j, name in enumerate(set_names):
			result[i, j] = calc_enrichment_score(ordered_genes, values, gene_sets[name], exponent=1.0)
	
	res_df = pd.DataFrame(result, index=expression_df.index, columns=set_names)
	return res_df


In [ ]:
# Prepare mRNA data for enrichment score calculation
# Assuming df_mrna has genes as rows (index) and samples as columns
# We need to transpose it so that samples are rows and genes are columns

# Check if we need to transpose
if df_mrna.shape[0] > df_mrna.shape[1]:
	# Likely: rows=genes, cols=samples -> transpose to rows=samples, cols=genes
	df_mrna_transposed = df_mrna.T
	print(f"Transposed mRNA data: {df_mrna_transposed.shape} (samples x genes)")
else:
	# Already in correct format: rows=samples, cols=genes
	df_mrna_transposed = df_mrna
	print(f"mRNA data: {df_mrna_transposed.shape} (samples x genes)")

df_mrna_transposed.head()

In [ ]:
# make the first row as header for df_mrna_transposed
df_mrna_transposed.columns = df_mrna_transposed.iloc[0]
df_mrna_transposed = df_mrna_transposed.iloc[1:]
df_mrna_transposed.head()

In [ ]:
# Calculate enrichment scores for all samples
# This may take a while depending on the number of samples and gene sets
print(f"Calculating enrichment scores for {df_mrna_transposed.shape[0]} samples and {len(gene_sets)} gene sets...")

enrichment_scores = get_functional_spectra(df_mrna_transposed, gene_sets, scale=False)

print(f"\nEnrichment scores shape: {enrichment_scores.shape}")
print(f"\nFirst few rows and columns:")
enrichment_scores.head()

In [ ]:
# Display summary statistics
print("Enrichment scores summary:")
print(enrichment_scores.describe())
print(f"\nSample of enrichment scores (first 5 samples, first 5 gene sets):")
print(enrichment_scores.iloc[:5, :5])

In [ ]:
# save to csv
enrichment_scores.to_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/BRCA_mRNA_enrichment_scores.csv')

In [ ]:
import pandas as pd
df_methy = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Original/BRCA_Methy.csv')

In [ ]:
df_cnv = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Original/BRCA_CNV.csv')

In [ ]:
df_mrna = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Original/BRCA_mRNA.csv')
df_mrna

In [ ]:
df_cnv

In [ ]:
df_methy

### Preprocess for training

In [ ]:
import pandas as pd 

df_cnv = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Top/BRCA_CNV_top.csv')
df_mrna = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Top/BRCA_mRNA_top.csv')
df_mirna = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Top/BRCA_miRNA_top.csv')
df_methy = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Top/BRCA_Methy_top.csv')

df_cnv.shape, df_mrna.shape, df_mirna.shape, df_methy.shape

In [ ]:
import pandas as pd

In [ ]:
df_label = pd.read_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Top/BRCA_label_num.csv')
df_label.shape

In [ ]:
y = df_label.iloc[:, 0].to_numpy()  # shape (N,)

In [ ]:
y

In [ ]:
df_label

In [ ]:
df_mirna

In [ ]:
df_mirna.dropna(inplace=True)
df_mirna.to_csv('/home3/tuannd/Cancer-Multi-Omics-Benchmark/Main_Dataset/Classification_datasets/GS-BRCA/Top/BRCA_miRNA_top_dropna.csv', index=False)